In [15]:
# Accessing Google API key
import os
from google.colab import userdata

try:
    Capstone_project_key = userdata.get("Capstone_project_key")
    os.environ["GOOGLE_API_KEY"] = Capstone_project_key # Set GOOGLE_API_KEY for genai library
    os.environ["Capstone_project_key"] = Capstone_project_key # Keep original for consistency if needed elsewhere
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'Capstone_project_key' to your Kaggle secrets. Details: {e}"
    )

✅ Gemini API key setup complete.


In [4]:
# Accessing Food API key: https://fdc.nal.usda.gov/api-guide
import os

try:
    Food_API = userdata.get("Food_API")
    os.environ["Food_API"] = Food_API
    print("✅ Food API key setup complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'Capstone_project_key' to your Kaggle secrets. Details: {e}"
    )

✅ Food API key setup complete.


In [1]:
#importing ADK components
from google.adk.agents import Agent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types


print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


In [78]:
#retry config to feed into Agent parameters
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)

In [98]:
#AlertAgent
from google.adk.tools import AgentTool
# AlertAgent = Agent(
#     name='AlertAgent',
#     model=Gemini(model='gemini-2.5-flash-lite', retry_options=retry_config),
#     instruction='You are an agent that sends alerts.',
#     tools=[]
# )
from datetime import datetime

def get_time() -> str:
    """Get the current time."""
    return datetime.now().isoformat()

Alert_Agent = Agent(
    name='AlertAgent',
    model=Gemini(model='gemini-2.5-flash-lite', retry_options=retry_config),
    description="You are a helpful Diabetes Assistant",
    instruction="""
    You are an Diabetes Alert Agent. Your task is to generate alerts for patients who are on medication to take their medication. 
    Use the given tool to determine the current time and generate alerts based on the following rules:
    For pre-meal medications, generate alerts at their breakfast, lunch, and dinner time. 
    For once a day medications, generate an alert at their preferred time of the day. 
    For users who take long acting insulin every night, generate an alert every night. 
    For users who take weekly GLP-1 Agonists, generate an alert every week at the patients preferred time of the week.""",
    tools=[get_time],
)

In [99]:
#GlucosePrediction
# This cell doesn't seem to define an agent, but rather a class and tool definition.
# If it were an agent, it would be defined here. For now, assuming it's correctly handled via the tool.


In [100]:
#InsulinAgent
from google.adk.tools import AgentTool
# InsulinAgent = Agent(
#     name='InsulinAgent',
#     model=Gemini(model='gemini-2.5-flash-lite', retry_options=retry_config),
#     instruction='You are an agent that provides insulin recommendations.',
#     tools=[]
# )

#InsulinAgent
def get_insulin_dose(glucose_level: int) -> dict:  
    """Returns the recommended insulin dose based on the glucose level."""
    if glucose_level < 151:
        return {"status": "success", "dose": "No insulin needed"}
    elif 151 <= glucose_level <= 200:
        return {"status": "success", "dose": "Take 2 units of short acting insulin before meal"}
    elif 201 <= glucose_level <= 250:
        return {"status": "success", "dose": "Take 4 units of short acting insulin before meal"}
    elif 251 <= glucose_level <= 300:
        return {"status": "success", "dose": "Take 6 units of short acting insulin before meal"}
    elif 301 <= glucose_level <= 350:
        return {"status": "success", "dose": "Take 8 units of short acting insulin before meal"}
    elif 351 <= glucose_level <= 400:
        return {"status": "success", "dose": "Take 10 units of short acting insulin before meal"}
    else:
        return {"status": "error", "message": "Glucose level too high, please call doctor"}

Insulin_Agent = Agent(
    name='InsulinRecommenderAgent',
    model=Gemini(model='gemini-2.5-flash-lite', retry_options=retry_config),
    description="You are an expert Diabetes coach. Given a patients glucose level, provide a suggestion of recommended dosage of insulin.",
    instruction="""
    You are an Insulin Recommender Agent. Your task is to provide a suggestion of appropriate insulin dosage based on the patient's glucose level
    so the patient can confirm with their doctor before taking any insulin. The patient will provide you will all of their current information. Find their current
    CGM Glucose reading and use the get_insulin_dose tool to determine correct dosage of insulin.""",
    tools=[get_insulin_dose],
)

In [101]:
# Tool: Food API
API_KEY = Food_API
import requests
#@trace_execution("search_food_by_carbs")
def search_food_by_carbs(food_name: str, max_carbs: float):

    #MealAgent_logger.info(f"Tool called: search_food_by_carbs | food={food_name} | max_carbs={max_carbs}")

    url = "https://api.nal.usda.gov/fdc/v1/foods/search"

    params = {
        "query": food_name,
        "pageSize": 20,
        "api_key": API_KEY
    }

    r = requests.get(url, params=params).json()

    foods = []

    for food in r["foods"]:
        nutrients = food.get("foodNutrients", [])
        nutrient_map = {n["nutrientName"]: n["value"] for n in nutrients}

        carbs = nutrient_map.get("Carbohydrate, by difference")
        protein = nutrient_map.get("Protein")
        calories = nutrient_map.get("Energy (kcal)") or nutrient_map.get("Energy")

        calories_from_carbs = carbs * 4 if carbs is not None else None

        if carbs is not None and carbs <= max_carbs:
            foods.append({
                "name": food["description"],
                "carbs_g": carbs,
                "protein_g": protein,
                "calories_kcal": calories,
                "calories_from_carbs": calories_from_carbs,
                "serving_size": food.get("servingSize"),
                "serving_unit": food.get("servingSizeUnit")
            })
    #MealAgent_logger.info(f"Tool result count: {len(foods)} foods returned")


    return foods

In [102]:
# testing
search_food_by_carbs("apple",20)

[{'name': 'APPLE',
  'carbs_g': 14.3,
  'protein_g': 0.0,
  'calories_kcal': 52.0,
  'calories_from_carbs': 57.2,
  'serving_size': 154.0,
  'serving_unit': 'g'},
 {'name': 'APPLE',
  'carbs_g': 11.7,
  'protein_g': 0.0,
  'calories_kcal': 46.0,
  'calories_from_carbs': 46.8,
  'serving_size': 240.0,
  'serving_unit': 'ml'},
 {'name': 'APPLE',
  'carbs_g': 14.0,
  'protein_g': 0.41,
  'calories_kcal': 54.0,
  'calories_from_carbs': 56.0,
  'serving_size': 242.0,
  'serving_unit': 'g'},
 {'name': 'Apple, raw',
  'carbs_g': 14.8,
  'protein_g': 0.17,
  'calories_kcal': 61,
  'calories_from_carbs': 59.2,
  'serving_size': None,
  'serving_unit': None},
 {'name': 'Apple cider',
  'carbs_g': 11.3,
  'protein_g': 0.1,
  'calories_kcal': 46,
  'calories_from_carbs': 45.2,
  'serving_size': None,
  'serving_unit': None},
 {'name': 'Apple juice, 100%',
  'carbs_g': 11.34,
  'protein_g': 0.09,
  'calories_kcal': 48,
  'calories_from_carbs': 45.36,
  'serving_size': None,
  'serving_unit': None},

In [103]:
#MealAgent

MealAgent =  Agent(
name="MealRecommenderAgent",
model=Gemini(
    model="gemini-2.5-flash-lite",
    api_key=Capstone_project_key,   # corrected: pass the variable, not the string literal
    retry_options=retry_config
),
# This instruction tells the Meal Agent HOW to use its tools (which are the other agents).
instruction="""

Role

You are a Diabetes Nutrition Coach Agent.
Your goal is to recommend meals and hydration strategies that help keep the user's blood glucose within 90–150 mg/dL.

Use food nutrition data from the USDA FoodData Central API to construct meal recommendations.Use search_food_by_carbs tool
to interact with the API.

Decision Rules
1. Hypoglycemia Management

If glucose < 70 mg/dL:

Recommend 15 g fast-acting carbohydrates (e.g., fruit juice or glucose tablets).

Ask the user to wait 15 minutes.

Recommend rechecking glucose levels.

If glucose remains below 70 mg/dL, repeat the 15 g carbohydrate treatment.

After glucose rises above 70 mg/dL, recommend a balanced meal.

2. Hyperglycemia Hydration

If glucose > 180 mg/dL:

Recommend drinking 500 mL–1 L of water to support hydration.

3. Meal Construction Rules

When recommending meals, present food items in the following order:

Protein

Vegetables

Carbohydrates

Meals should:

Include lean protein

Include non-starchy vegetables

Include controlled carbohydrate portions

4. Carbohydrate Impact Assumption

Assume:

10 g of carbohydrates may increase blood glucose by approximately 30–50 mg/dL.

Use this assumption to estimate appropriate carbohydrate quantities for the desired change in the blood glucose.

5. Medication Timing Rule

If the user has recently taken insulin or oral diabetes medication, recommend meals 15 minutes after medication intake when appropriate.

Tool Usage

To generate food recommendations:

use search_food_by_carbs tool.

The tool should retrieve:

food name

macronutrients

carbohydrate content

serving size

Select foods that satisfy the meal rules.

Output Format

Provide recommendations in the following format:

Glucose Status:
(normal / high / low)

Hydration Recommendation:
(if applicable)

Meal Recommendation

Protein:

example food + portion

Vegetables:

example food + portion

Carbohydrates:

example food + portion

Estimated Carbohydrates:
XX grams

Estimated impact of glucose level:
This meal provides an expected XX mg/dL change to bring blood glucose within the target range of 90–150 mg/dL.

Additional Guidance:
(e.g., recheck glucose after 15 minutes)
"""
, tools = [search_food_by_carbs]
, output_key="MealAgent_recommendation",
)
#MealAgent_logger.info(f"MealAgent Output: {MealAgent_recommendation}")
#MealAgent = create_meal_agent()
print("✅ MealAgent created.")

✅ MealAgent created.


In [104]:
## Testing: Executing MealAgent
#Runner to run the Main Agent
#from google.adk.plugins.logging import LoggingPlugin
from google.adk.plugins.logging_plugin import (
    LoggingPlugin,)
runner_Meal = InMemoryRunner(agent=Alert_Agent
                       ,plugins=[
                                    LoggingPlugin()
                                ],
                       ) # <---- 2. Add the plugin. Handles standard Observability logging across ALL agents)

print("✅ Runner created.")


2026-03-22 14:37:32,477 - INFO - google_adk.google.adk.plugins.plugin_manager - Plugin 'logging_plugin' registered.


✅ Runner created.


In [105]:
# Testing: Running the MealAgent


response = await runner_Meal.run_debug(
"""
min_past = minimum of CGM reading in the past 2 hours in 5 min interval =60
max_past = minimum of CGM reading in the past 2 hours in 5 min interval = 175

current CGM reading= 143
last time when the meal was taken: 1 PM ET

min_predicted = minimum of CGM reading in the next 1 hours in 5 min interval = 65
max_predicted = maximum of CGM reading in the next 1 hours in 5 min interval = 200

Weight	=[75]		# in KG
Height	=[1.65]		# in meter
Diet Preference = [vegan] # vegan, veg, fruits, non-veg etc.]

Usual meal time	= [breakfast: 7AM ET , lunch 3:30 ET: , dinner: 6PM ET ]	#breakfast, lunch, Dinner timings

Oral Medications dosing	= [before all 3 meals] #pre-meal/once a day/once a week. What time?

insulin intake =[Y] #YN
GLP-1 Agonist intake = [N] # YN->when

"""
)

2026-03-22 14:37:32,582 - INFO - google_adk.google.adk.models.google_llm - Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False



 ### Created new session: debug_session_id

User > 
min_past = minimum of CGM reading in the past 2 hours in 5 min interval =60
max_past = minimum of CGM reading in the past 2 hours in 5 min interval = 175

current CGM reading= 143
last time when the meal was taken: 1 PM ET

min_predicted = minimum of CGM reading in the next 1 hours in 5 min interval = 65
max_predicted = maximum of CGM reading in the next 1 hours in 5 min interval = 200

Weight	=[75]		# in KG
Height	=[1.65]		# in meter
Diet Preference = [vegan] # vegan, veg, fruits, non-veg etc.]

Usual meal time	= [breakfast: 7AM ET , lunch 3:30 ET: , dinner: 6PM ET ]	#breakfast, lunch, Dinner timings

Oral Medications dosing	= [before all 3 meals] #pre-meal/once a day/once a week. What time?

insulin intake =[Y] #YN
GLP-1 Agonist intake = [N] # YN->when


[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-a16cd294-c324-4c3e-8fd4-499164d3d0c5
[logging_plugin]    Session ID: debug_session_id
[logging_plugin]

2026-03-22 14:37:35,297 - INFO - google_adk.google.adk.models.google_llm - Response received from the model.
2026-03-22 14:37:35,300 - INFO - google_adk.google.adk.models.google_llm - Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False


[logging_plugin] 🧠 LLM RESPONSE
[logging_plugin]    Agent: AlertAgent
[logging_plugin]    Content: function_call: get_time
[logging_plugin]    Token Usage - Input: 440, Output: 10
[logging_plugin] 📢 EVENT YIELDED
[logging_plugin]    Event ID: a43e2e05-9623-493a-81ff-a7282c9078b2
[logging_plugin]    Author: AlertAgent
[logging_plugin]    Content: function_call: get_time
[logging_plugin]    Final Response: False
[logging_plugin]    Function Calls: ['get_time']
[logging_plugin] 🔧 TOOL STARTING
[logging_plugin]    Tool Name: get_time
[logging_plugin]    Agent: AlertAgent
[logging_plugin]    Function Call ID: 28syywwk
[logging_plugin]    Arguments: {}
[logging_plugin] 🔧 TOOL COMPLETED
[logging_plugin]    Tool Name: get_time
[logging_plugin]    Agent: AlertAgent
[logging_plugin]    Function Call ID: 28syywwk
[logging_plugin]    Result: 2026-03-22T14:37:35.299202
[logging_plugin] 📢 EVENT YIELDED
[logging_plugin]    Event ID: 0870a2ea-983e-408d-a2b6-be2bd25177f0
[logging_plugin]    Author: Ale

2026-03-22 14:37:48,405 - INFO - google_adk.google.adk.models.google_llm - Response received from the model.


[logging_plugin] 🧠 LLM RESPONSE
[logging_plugin]    Agent: AlertAgent
[logging_plugin]    Content: text: 'Hello! Based on the current time and your personalized schedule, here is your alert:

**Medication Reminder:**
*   **Lunch Medication:** You are scheduled for lunch at **3:30 PM ET**. Please remember ...'
[logging_plugin]    Token Usage - Input: 885, Output: 180
[logging_plugin] 📢 EVENT YIELDED
[logging_plugin]    Event ID: ee5937ce-b051-4bf8-b280-ffbd04a5cc07
[logging_plugin]    Author: AlertAgent
[logging_plugin]    Content: text: 'Hello! Based on the current time and your personalized schedule, here is your alert:

**Medication Reminder:**
*   **Lunch Medication:** You are scheduled for lunch at **3:30 PM ET**. Please remember ...'
[logging_plugin]    Final Response: True
AlertAgent > Hello! Based on the current time and your personalized schedule, here is your alert:

**Medication Reminder:**
*   **Lunch Medication:** You are scheduled for lunch at **3:30 PM ET**. Please rememb

In [106]:
# Wrapping MealAgent into trace_execution wrapper
#@trace_execution("MealAgent")
#def get_meal_recommendation(user_input):
#    return MealAgent.run(user_input)

In [107]:
#ExerciseAgent
from google.adk.tools import AgentTool
import pandas as pd

# ExerciseAgent = Agent(
#     name='ExerciseAgent',
#     model=Gemini(model='gemini-2.5-flash-lite', retry_options=retry_config),
#     instruction='You are an agent that provides exercise recommendations.',
#     tools=[]
# )

def get_exercise_intensity(glucose_level:int) -> list:
    """Returns the recommended exercise intensity based on the glucose level."""
    if glucose_level < 90:
        return ["Avoid"]
    elif 90 <= glucose_level <= 124:
        return ["Light"]
    elif 125 <= glucose_level <= 180:
        return ["Light", "Moderate", "Vigorous"]
    elif 181 <= glucose_level <= 270:
        return ["Light", "Moderate"]
    elif glucose_level > 270:
        return ["Avoid"]

def search_exercise_by_intensity(intensity: str) -> list:
    df = pd.read_csv("traincalc-met-values-latest.csv")
    filtered = df[df["Intensity"] == intensity]
    return filtered[["Description", "MET"]].to_dict(orient="records")

def get_exercise_recommendation(glucose_level: int, meal_carbs: int = None) -> dict:
    # calculating exercise based on current glucose, next steps incorporate pre/post meal exercise recommendations based on carbs in the meal
    intensity_levels = get_exercise_intensity(glucose_level)

    if "Avoid" in intensity_levels:
        return {
            "status": "unsafe",
            "message": "Glucose level not suitable for exercise",
            "pre_exercise": "Consume carbohydrates and recheck glucose"
        }

    all_exercises = []

    for level in intensity_levels:
        exercises = search_exercise_by_intensity(level)

        for e in exercises:
            all_exercises.append({
                "description": e["Description"],
                "met": e["MET"],
                "intensity": level
            })

    return {
        "status": "ok",
        "recommended_exercises": all_exercises
    }

Exercise_Agent = Agent(
    name='ExerciseRecommenderAgent',
    model=Gemini(model='gemini-2.5-flash-lite', retry_options=retry_config),
    description="Given glucose level, recommend appropriate exercise",
    instruction="""
    You are an expert Diabetes Exercise Coach. Your task is to recommend appropriate exercises based on the patient's glucose level. Use the provided get_exercise_recommendation tool 
    to determine suitable exercises based on the glucose level. Then give a recommendation of exercise types, specific exercises, and duration that the patient 
    can do to help manage their blood glucose levels effectively. Be sure to consider the patient's safety and recommend NO exercise if glucose levels are too low or too high. 
    Use the following guidelines for your recommendations:
    Lower than 90 mg/dL (5.0 mmol/L). Your blood sugar may be too low to exercise safely. Before you work out, have a small snack that includes 15 to 30 grams of carbohydrates. 
    Some examples are fruit juice, fruit and crackers. Or take 10 to 20 grams of glucose products, which come in forms such as gels, powders and tablets. 
    After you exercise, check your blood sugar again to find out if it's about 90 mg/dL.
    90-124 mg/dL (5-6.9 mmol/L). Take 10 grams of glucose before you exercise.
    126-180 mg/dL (7-10 mmol/L). You're ready to exercise. But be aware that blood sugar may rise if you do strength training. 
    Blood sugar also may rise if you do short bursts of hard aerobic exercise known as high-intensity interval training.
    182-270 mg/dL (10.2-15 mmol/L). It's okay to exercise. Be aware that blood sugar may rise if you do strength training or high-intensity interval training.
    Over 270 mg/dL (15 mmol/L). This is a caution zone. Your blood sugar may be too high to exercise safely. 
    Before you work out, test your urine for substances called ketones. The body makes ketones when it breaks down fat for energy. 
    The presence of ketones suggests that your body doesn't have enough insulin to control your blood sugar.""",
    tools=[get_exercise_recommendation],
)

In [108]:
#SafetyGuardAgent
# This agent is not yet used in the Main_agent definition, so no change needed here for now.

In [109]:
## Creating a dummy prediction model
# input: <min_past and max_past that automatically create random 24 past CGM data points within min & max range
# output: <min_pred, max_pred and randomly generated 12 future CGM data points within min and max range

import random

class DummyCGMModel:

    def predict(self, min_past, max_past):

        # Generate 24 past CGM values
        past_cgm = [random.randint(min_past, max_past) for _ in range(24)]

        # Define prediction range
        min_pred = min(past_cgm) - random.randint(5, 15)
        max_pred = max(past_cgm) + random.randint(5, 15)

        # Generate 12 future predictions
        future_cgm = [random.randint(min_pred, max_pred) for _ in range(12)]

        return {
            "past_cgm_24_points": past_cgm,
            "min_pred": min_pred,
            "max_pred": max_pred,
            "future_cgm_12_points": future_cgm
        }

# Save the model with joblib
import joblib

model = DummyCGMModel()

joblib.dump(model, "dummy_cgm_model.joblib")

print("Model saved!")



Model saved!


In [110]:
# prediction tool for the Main agent
def predict_glucose(min_past,max_past):

    model = joblib.load("dummy_cgm_model.joblib")
    prediction = model.predict(min_past, max_past) # min max of past 24 values

    return prediction

In [111]:
# Root Coordinator: Orchestrates the workflow by calling the sub-agents as tools.
#MainAgent_logger.info("Initializing Main agent")
Main_agent = Agent(
    name="Orchestrator_Agent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # This instruction tells the root agent HOW to use its tools (which are the other agents).
    instruction="""
    System Role

You are a Blood Glucose Coaching Orchestrator Agent that helps users with diabetes maintain their blood glucose within the target range of 90–150 mg/dL.

Your job is to coordinate multiple specialized agents and tools to generate safe, personalized recommendations.

You do not guess medical information. Instead, you call the appropriate tools and synthesize their results into clear recommendations.

Primary Objective

Maintain the user’s glucose within 90–150 mg/dL by coordinating:

glucose prediction using glucose prediction tool

medication reminders

insulin dosing guidance

meal recommendations

exercise recommendations

hydration advice

Workflow (You MUST follow this order)

1. **Extract Key Information:** From the user's input, carefully extract the numerical values for `min_past` and `max_past` from the lines describing CGM readings. Also, extract the `current CGM reading`, `last time when the meal was taken`, `Current time`, `Weight`, `Height`, `Diet Preference`, `Usual meal time`, `Oral Medications dosing`, and `insulin intake`.

2. **Step 1 — Medication Alert:**
   - Check if the `Current time` corresponds to a `medication schedule` (e.g., 'pre-meal-before all 3 meals' and if a meal is upcoming or recently passed). If the `Oral Medications dosing` is 'pre-meal-before all 3 meals' and `Current time` is close to any `Usual meal time`.
   - If a medication is due, YOU MUST call the `AlertAgent` tool to notify the user about: oral medication, insulin, GLP-1 agonist. Include specific medication type if applicable (e.g., 'oral medication before lunch').

3. **Step 2 — Predict Future Glucose:**
   - YOU MUST call the `predict_glucose` tool using the extracted numerical `min_past` and `max_past` values as inputs.

4. **Step 3 — Insulin Recommendation:**
   - If `insulin intake` is 'Y':
   - YOU MUST call the `InsulinAgent` tool.
   - Provide `current CGM reading` OR `predicted glucose at the time of the meal` (from Step 2's prediction) as input to the `InsulinAgent`tool.
   - InsulinAgent`tool provides a dosage recommendation based on the glucose level 15 minutes before the preferred meal time.

5. **Step 4 — Meal Recommendation:**
   - If the `Current time` is close to the user’s `Usual meal time` (breakfast, lunch, or dinner) OR if the `current CGM reading` indicates a need for immediate meal intervention (e.g., low glucose):
   - YOU MUST call the `MealAgent` tool.
   - Provide the `predicted glucose trajectory` (from Step 2), `Diet Preference`, `current CGM reading`, and `glucose target range` (90–150 mg/dL) as input.
   - The meal recommendation should help keep predicted glucose within 90–150 mg/dL.
   - Include hydration guidance if appropriate, based on `current CGM reading`.

6. **Step 5 — Exercise Recommendation:**
   - YOU MUST call the `ExerciseAgent` tool to generate an exercise recommendation based on the required Calories burn to keep the glucose in the targetted range.
   - Use: `predicted glucose trend` (from Step 2), `time since last meal`, and `safety constraints` (e.g., avoid exercise if glucose < 70 mg/dL).
   - Exercise recommendations should help maintain glucose within the target range.

Safety Rules (Always Follow)

Avoid recommending exercise if glucose is < 70 mg/dL.

Hypoglycemia protocol

If glucose < 70 mg/dL:

Eat fast-acting carbohydrates first

Wait 15 minutes

Recheck glucose

Repeat if still low

Medication timing

Pre-meal medication should be taken 15 minutes before the meal.

Exercise timing

Post-meal exercise should typically occur ~2 hours after eating.

Tool Usage Rules

You must follow these rules:

Do not fabricate glucose predictions.

Do not fabricate insulin dosage.

Do not invent nutritional values.

Always use the appropriate tool when required.

Tools available:

AlertAgent

predict_glucose

InsulinAgent

MealAgent

ExerciseAgent

Final Response Format

After completing all tool calls, present a clear summary to the user containing:

1. Glucose Outlook

Brief description of past, current, and predicted glucose trend. Show relavant numbers in table.
Include user inputs - `last time when the meal was taken`, `Current time`, `Weight`, `Height`, `Diet Preference`, `Usual meal time`, `Oral Medications dosing`, and `insulin intake`.

2. Medication Reminder

If applicable.

3. Meal Recommendation

Suggested meal and hydration.

4. Exercise Recommendation

Activity type and timing.

5. Safety Notes

Any warnings about hypoglycemia or high glucose.

Keep explanations simple, supportive, and actionable.
    """,
    # We wrap the sub-agents in `AgentTool` to make them callable tools for the root agent.
    tools=[AgentTool(Alert_Agent), FunctionTool(predict_glucose), AgentTool(Insulin_Agent), AgentTool(MealAgent), AgentTool(Exercise_Agent)],
    # For the MainAgent, ensure that sub_agents are correctly defined before use
    sub_agents=[Alert_Agent, Insulin_Agent, MealAgent, Exercise_Agent],
    output_key="MainAgent_output"
)
#MainAgent_logger.info(f"MainAgent Output: {MainAgent_output}")


print("✅ Main_agent created.")

✅ Main_agent created.


In [112]:
#Runner to run the Main Agent
from google.adk.plugins.logging_plugin import (
    LoggingPlugin,)
import logging

# Configure basic logging to include timestamp and write to file
log_file_name = "agent_run.log"
logging.basicConfig(
    level=logging.INFO, # Set desired logging level
    format='%(asctime)s - %(levelname)s - %(name)s - %(message)s',
    handlers=[
        logging.StreamHandler(), # Also log to console
        logging.FileHandler(log_file_name) # Log to a file
    ]
)

runner = InMemoryRunner(agent=Main_agent
                       ,plugins=[
                                    LoggingPlugin() # LoggingPlugin will now use the globally configured logger
                                ],
                       )
# <---- 2. Add the plugin. Handles standard Observability logging across ALL agents)

print("✅ Runner created. Logs will be saved to " + log_file_name)


2026-03-22 14:37:48,452 - INFO - google_adk.google.adk.plugins.plugin_manager - Plugin 'logging_plugin' registered.


✅ Runner created. Logs will be saved to agent_run.log


In [ ]:
# Provide user input to execute the main agent response
import time
# from google.genai.types import TokenUsage # Ensure this is imported - REMOVED THIS LINE

# Pricing per 1K tokens for Gemini models (example values, check official pricing)
PRICING = {
    "gemini-2.5-pro": {
        "input": 0.0035,
        "output": 0.0105
    },
    "gemini-2.5-flash": {
        "input": 0.000125,
        "output": 0.000375
    }
}

# Initialize token counters per model
token_usage_by_model = {}

# Execute the agent and measure time
start_time = time.time()
response = await runner.run_debug(
"""
min_past = minimum of CGM reading past 2 hours in 5 min interval =60
max_past = minimum of CGM reading past 2 hours in 5 min interval = 200
current CGM reading = 165
last meal taken: Lunch at 1 PM ET
Current time= 1:30 PM ET
Weight	=[75]		# in KG
Height	=[1.65]		# in meter
Diet Preference = ['Non-Veg'] # vegan, veg, fruits, non-veg etc.

Usual meal time	= [breakfast: 7 AM ET , lunch: 12:30 PM ET, dinner: 7 PM ET ]	#breakfast, lunch, Dinner timings

Oral Medications dosing	= [pre-meal-before all 3 meals] #pre-meal/once a day/once a week. What time?

insulin intake =[Y] #YN
GLP-1 Agonist intake = [N] # YN->when

"""
)
end_time = time.time()
elapsed_time = end_time - start_time
print(f"✅ Main_agent took {elapsed_time:.2f} seconds to generate the response.")

# Process response to get token usage and calculate cost
total_estimated_cost = 0.0
total_input_tokens = 0
total_output_tokens = 0

for event in response:
    # Check if the event has metadata, token_usage, and a model_version
    if hasattr(event, 'metadata') and event.metadata and \
       hasattr(event.metadata, 'token_usage') and event.metadata.token_usage and \
       hasattr(event, 'model_version') and event.model_version:

        token_usage = event.metadata.token_usage
        model_name = event.model_version

        if model_name not in token_usage_by_model:
            token_usage_by_model[model_name] = {"input": 0, "output": 0}

        token_usage_by_model[model_name]["input"] += token_usage.input_tokens
        token_usage_by_model[model_name]["output"] += token_usage.output_tokens

        total_input_tokens += token_usage.input_tokens
        total_output_tokens += token_usage.output_tokens

# Calculate cost for each model and accumulate total cost
for model_name, usage in token_usage_by_model.items():
    if model_name in PRICING:
        input_cost_per_k = PRICING[model_name]["input"]
        output_cost_per_k = PRICING[model_name]["output"]

        model_cost = (usage["input"] / 1000 * input_cost_per_k) + \
                     (usage["output"] / 1000 * output_cost_per_k)
        total_estimated_cost += model_cost
        print(f"Model: {model_name}, Input Tokens: {usage['input']}, Output Tokens: {usage['output']}, Cost: ${model_cost:.6f}")
    else:
        print(f"Warning: Pricing not available for model '{model_name}'. Tokens used: Input={usage['input']}, Output={usage['output']}.")

print(f"Total Input Tokens (Across all models): {total_input_tokens}")
print(f"Total Output Tokens (Across all models): {total_output_tokens}")
print(f"Total Estimated Cost (USD): ${total_estimated_cost:.6f}")


# Save the final agent output to a file
output_file_name = "agent_final_output.txt"

with open(output_file_name, "w") as f:
    # Assuming the final output is in the last event of the response
    # or can be constructed from the events.
    # This example extracts text from content parts of the last event.
    # Adjust this logic based on the actual structure of your agent's final output.
    if response and isinstance(response, list):
        final_event = response[-1] # Get the last event
        if hasattr(final_event, 'content') and final_event.content and final_event.content.parts:
            for part in final_event.content.parts:
                if hasattr(part, 'text'):
                    f.write(part.text)
                elif hasattr(part, 'function_call'): # Handle function calls if they are part of final output
                    f.write(f"Function Call: {part.function_call.name}\n")
                    f.write(f"Arguments: {part.function_call.args}\n")
    elif response: # If response is not a list but has a direct text output
        f.write(str(response))
    else:
        f.write("No output from agent.")
    f.write(f"\n\nTime taken by Main_agent: {elapsed_time:.2f} seconds")
    f.write(f"\nTotal Input Tokens: {total_input_tokens}")
    f.write(f"\nTotal Output Tokens: {total_output_tokens}")
    f.write(f"\nEstimated Cost (USD): ${total_estimated_cost:.6f}")


print(f"✅ Agent final output saved to {output_file_name}")